# Frozen Lake 16x16
Modified the credit assignment to NOT overwrite rewards that are bigger. This was a flaw in the code which made the agent loop around over and over after obtaining a gift.

imports

In [100]:
import gymnasium as gym
import sys

print(f"python: {sys.version[:7]}")
print(f"gym: {gym.__version__}")

python: 3.10.18
gym: 1.2.0


init env

In [127]:
from gymnasium.envs.toy_text.frozen_lake import generate_random_map
import gymnasium as gym

map = generate_random_map(size=16)

env = gym.make(
    'FrozenLake-v1',
    render_mode="rgb",
    map_name="16x16", 
    is_slippery=False,
    desc=map
)

c:\Users\thezo\anaconda3\envs\frozenlake\lib\site-packages\gymnasium\envs\registration.py:728: UserWarning: WARN: The environment is being initialised with render_mode='rgb' that is not in the possible render_modes (['human', 'ansi', 'rgb_array']).
  logger.warn(


initialize q_table

In [128]:
q_table = [[0 for _ in range(env.action_space.n)] for _ in range(env.observation_space.n)]

start learning
- usually there are max 500 steps before it truncates. This is manually lifted up to 1000 steps.
- also modified the credit assignment, i now make sure i dont overwrite a reward that is bigger.

In [129]:
import gymnasium as gym
import time
import random

def get_opposite_direction_action(action):
    opposite_actions = {0: 2, 1: 3, 2: 0, 3: 1}
    return opposite_actions[action]

env = gym.make(
    'FrozenLake-v1',
    render_mode="rgb",
    map_name="16x16", 
    is_slippery=False,
    desc=map
)

# parameters
hard_break = False
epochs = 20

for epoch in range(epochs):
    if hard_break:
        break

    epsilon = 1 # exploration rate
    epsilon_decay = 0.99995
    
    for episode in range(500):
        if hard_break:
            break
        observation, _ = env.reset()
        action_history = []
        observation_history = []
        previous_reward = 0
        credit_decay = 0.999
        
        for step in range(1000):

            # choose action
            if random.uniform(0, 1) < epsilon or previous_reward < 0: # exploration if - episilon or last reward was negative
                possible_actions = q_table[observation].copy()
                
                action = env.action_space.sample() # random action - (that doesnt go in hole)
                while q_table[observation][action] == -2:
                    action = env.action_space.sample() # (random action)
            else:
                possible_actions = q_table[observation].copy()
                if action_history:
                    last_action = action_history[-1] # get last action
                    possible_actions[get_opposite_direction_action(last_action)] = float('-inf') # lock opposite direction action - prevents moving back
                highest_reward = max(possible_actions) # (greedy action)
                action = possible_actions.index(highest_reward)
            # action = int(input())

            # step
            new_observation, reward, terminated, truncated, info = env.step(action)
            observation_history.append(observation)
            action_history.append(action)

            # Reward Shaping
            if reward == 1:
                print("Gift Reached!")
                mod_reward = reward
                for observation, action in zip(reversed(observation_history), reversed(action_history)):
                    if q_table[observation][action] < mod_reward: # only overwrite if its smaller, so we dont get lost in infinite loops

                        q_table[observation][action] = mod_reward # we dont use rewards anymore since this can result in agent looping
                        mod_reward = round(mod_reward * credit_decay, ndigits=8)
                hard_break = True
                break

            else:
                if reward < 1 and terminated: # (dead)
                    reward = -2
                    q_table[observation][action] = reward
                elif observation == new_observation: # (stuck)
                    reward = -1
                    q_table[observation][action] = reward


            observation = new_observation
            epsilon = epsilon * epsilon_decay
            previous_reward = reward

            # time.sleep(1)
            step+=1

            if terminated:
                break
    print(f"epoch: {epoch + 1} ended")


epoch: 1 ended
epoch: 2 ended
epoch: 3 ended
epoch: 4 ended
epoch: 5 ended
epoch: 6 ended
epoch: 7 ended
epoch: 8 ended
epoch: 9 ended
epoch: 10 ended
Gift Reached!
epoch: 11 ended


test the agent

In [130]:
import gymnasium as gym
import time
import random

def get_opposite_direction_action(action):
    opposite_actions = {0: 2, 1: 3, 2: 0, 3: 1}
    return opposite_actions[action]

env = gym.make(
    'FrozenLake-v1',
    render_mode="human",
    map_name="16x16", 
    is_slippery=False,
    desc=map
)

# parameters
epsilon = 0 # exploration rate
epsilon_decay = 0.99
for episode in range(1):
    observation, _ = env.reset()
    action_history = []
    observation_history = []
    previous_reward = 0
    credit_decay = 0.98
    while True:

        # choose action
        if random.uniform(0, 1) < epsilon or previous_reward < 0: # exploration if - episilon or last reward was negative
            action = env.action_space.sample() # (random action)
            print('random')
        else:
            possible_actions = q_table[observation].copy()
            if action_history:
                last_action = action_history[-1] # get last action
                possible_actions[get_opposite_direction_action(last_action)] = float('-inf') # lock opposite direction action - prevents moving back
            highest_reward = max(possible_actions) # (greedy action)
            action = possible_actions.index(highest_reward)

        # step
        new_observation, reward, terminated, truncated, info = env.step(action)
        observation_history.append(observation)
        action_history.append(action)

        # Reward Shaping
        if reward == 1:
            print("Gift Reached!")
            mod_reward = reward
        else:
            if reward < 1 and terminated: # (dead)
                reward = -2
            elif observation == new_observation: # (stuck)
                reward = -1

        observation = new_observation
        epsilon = epsilon * epsilon_decay
        previous_reward = reward

        # time.sleep(1)
        
        if terminated or truncated:
            print(f'terminated: {terminated}')
            print(f'truncated: {truncated}')
            break


Gift Reached!
terminated: True
truncated: False


Succes!